# Stock Comparison Tool

This project compares several stocks over the same time period and ranks them by total return.

The aim is simple: use Python to collect market data, calculate returns consistently, and visualise which stock performed best over the selected dates.


## BLOCK 0 - IMPORTS

Purpose:
Import the libraries needed for downloading data, creating a results table, and plotting charts.


In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt


## BLOCK 1 - USER INPUTS

Purpose:
Choose the stocks and date range to compare.

The tickers should be separated by commas, for example: AAPL, MSFT, NVDA.


In [ ]:
tickers_input = input("Enter stock tickers separated by commas (e.g. AAPL, MSFT, NVDA): ")
start_date = input("Enter start date (YYYY-MM-DD): ")
end_date = input("Enter end date (YYYY-MM-DD): ")

tickers = [ticker.strip().upper() for ticker in tickers_input.split(",") if ticker.strip()]

print("Tickers selected:", tickers)
print("Start date:", start_date)
print("End date:", end_date)


## BLOCK 2 - DATA COLLECTION AND RETURN CALCULATION

Purpose:
Download each stock's price data and calculate total return.

Formula:
Total Return = (Final Price - Initial Price) / Initial Price


In [ ]:
results = []
price_history = pd.DataFrame()

for ticker in tickers:
    data = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False
    )

    if data.empty:
        print(f"No data found for {ticker}")
        continue

    prices = data["Close"].dropna()

    if isinstance(prices, pd.DataFrame):
        prices = prices.iloc[:, 0]

    if len(prices) < 2:
        print(f"Not enough data for {ticker}")
        continue

    initial_price = prices.iloc[0]
    final_price = prices.iloc[-1]
    total_return = (final_price - initial_price) / initial_price

    results.append({
        "Ticker": ticker,
        "Initial Price": initial_price,
        "Final Price": final_price,
        "Total Return": total_return
    })

    price_history[ticker] = prices

results_df = pd.DataFrame(results)
results_df


## BLOCK 3 - PERFORMANCE RANKING

Purpose:
Rank the stocks from highest return to lowest return.

This answers the main question: which stock performed best during the chosen period?


In [ ]:
if results_df.empty:
    print("No valid stock data was found. Try different tickers or dates.")
else:
    ranking_df = results_df.sort_values("Total Return", ascending=False).reset_index(drop=True)
    ranking_df["Rank"] = ranking_df.index + 1
    ranking_df = ranking_df[["Rank", "Ticker", "Initial Price", "Final Price", "Total Return"]]
    ranking_df


## BLOCK 4 - BEST AND WORST PERFORMERS

Purpose:
Identify the strongest and weakest stock in the comparison.


In [ ]:
if not results_df.empty:
    best_stock = ranking_df.iloc[0]
    worst_stock = ranking_df.iloc[-1]

    print(f"Best performing stock: {best_stock['Ticker']} ({best_stock['Total Return']:.2%})")
    print(f"Worst performing stock: {worst_stock['Ticker']} ({worst_stock['Total Return']:.2%})")


## BLOCK 5 - RETURN BAR CHART

Purpose:
Visualise total return for each stock.

The chart makes it easier to compare performance quickly.


In [ ]:
if not results_df.empty:
    chart_data = ranking_df.set_index("Ticker")["Total Return"]

    plt.figure(figsize=(9, 5))
    bars = plt.bar(chart_data.index, chart_data.values)
    plt.axhline(0, color="black", linewidth=1)
    plt.title(f"Stock Return Comparison: {start_date} to {end_date}")
    plt.ylabel("Total Return")

    for bar in bars:
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value,
            f"{value:.1%}",
            ha="center",
            va="bottom" if value >= 0 else "top"
        )

    plt.show()


## BLOCK 6 - NORMALISED PRICE CHART

Purpose:
Show how each stock moved over time after starting at 1.

This is useful because two stocks can end with similar returns but take very different paths.


In [ ]:
if not price_history.empty:
    normalised_prices = price_history / price_history.iloc[0]

    normalised_prices.plot(figsize=(11, 6))
    plt.title("Normalised Price Performance")
    plt.ylabel("Growth of $1")
    plt.xlabel("Date")
    plt.show()


## BLOCK 7 - CONCLUSION

Purpose:
Summarise what the tool does and what it does not do.

This project identifies which selected stock had the highest total return over the chosen period. It is useful for basic comparison, but it does not measure risk, valuation, or whether the stock is a good future investment.
